# Sessao 3 - Memory + Copa do Mundo (Didatico)

Este notebook faz 4 coisas, em ordem simples:
1. Cria conversas com `memory_subject_id` para testar memoria entre conversas
2. Semeia contexto de Copa do Mundo e valida short-term + long-term
3. Mantem o codigo enxuto, sem blocos de excecao desnecessarios
4. No final, usa DuckDuckGo como tool da LLM via OpenAI-compatible


In [1]:
!pip install openai python-dotenv httpx oci_genai_auth ddgs


  Using cached fake_useragent-2.2.0-py3-none-any.whl.metadata (17 kB)
  Using cached h2-4.3.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached socksio-1.0.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached hpack-4.1.0-py3-none-any.whl.metadata (4.6 kB)
Using cached fake_useragent-2.2.0-py3-none-any.whl (161 kB)
Using cached socksio-1.0.0-py3-none-any.whl (12 kB)
   ---------------------------------------- 0.0/4.7 MB ? eta -:--:--
   ---------------------------------------- 4.7/4.7 MB 23.5 MB/s eta 0:00:00
Using cached h2-4.3.0-py3-none-any.whl (61 kB)
Using cached hpack-4.1.0-py3-none-any.whl (34 kB)
Using cached hyperframe-6.1.0-py3-none-any.whl (13 kB)
  Attempting uninstall: primp
    Found existing installation: primp 0.15.0
    Uninstalling primp-0.15.0:
      Successfully uninstalled primp-0.15.0


  You can safely remove it manually.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Users\Amanda Machado\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1) Carregar variaveis de ambiente e escolher autenticacao por regiao

`load_dotenv()` continua ativo. A regra fica assim:
- Se `OCI_REGION=sa-saopaulo-1`: usa IAM com arquivo `config` (User Principal)
- Se `OCI_REGION=us-chicago-1`: usa API key
- Outras regioes: usa API key por padrao


In [2]:
import os
import json
import time
import httpx
from ddgs import DDGS
from dotenv import load_dotenv
from openai import OpenAI
from oci_genai_auth import OciUserPrincipalAuth

load_dotenv()

OCI_REGION = os.getenv("OCI_REGION", "")
OCI_OPENAI_API_KEY = os.getenv("OCI_OPENAI_API_KEY", "")
PROJECT_ID = os.getenv("PROJECT_ID", "")
MODEL_ID = os.getenv("MODEL_ID", "openai.gpt-5.2")
OCI_PROFILE = os.getenv("OCI_PROFILE", "DEFAULT")
OCI_CONFIG_FILE = os.getenv("OCI_CONFIG_FILE", "./config")
MEMORY_SUBJECT_ID = os.getenv("MEMORY_SUBJECT_ID", "copa-user-001")

assert OCI_REGION, "Defina OCI_REGION no .env"
assert PROJECT_ID, "Defina PROJECT_ID no .env"

BASE_URL = f"https://inference.generativeai.{OCI_REGION}.oci.oraclecloud.com/openai/v1"

if OCI_REGION == "sa-saopaulo-1":
    client = OpenAI(
        base_url=BASE_URL,
        api_key="not-used",
        project=PROJECT_ID,
        http_client=httpx.Client(auth=OciUserPrincipalAuth(profile_name=OCI_PROFILE, config_file=OCI_CONFIG_FILE)),
    )
    AUTH_MODE = "IAM_USER_PRINCIPAL"
else:
    assert OCI_OPENAI_API_KEY, "Defina OCI_OPENAI_API_KEY no .env"
    client = OpenAI(
        api_key=OCI_OPENAI_API_KEY,
        base_url=BASE_URL,
        project=PROJECT_ID,
    )
    AUTH_MODE = "GENAI_API_KEY"

print("BASE_URL:", BASE_URL)
print("OCI_REGION:", OCI_REGION)
print("AUTH_MODE:", AUTH_MODE)
print("PROJECT_ID:", PROJECT_ID)
print("MODEL_ID:", MODEL_ID)
print("MEMORY_SUBJECT_ID:", MEMORY_SUBJECT_ID)



BASE_URL: https://inference.generativeai.us-chicago-1.oci.oraclecloud.com/openai/v1
OCI_REGION: us-chicago-1
AUTH_MODE: GENAI_API_KEY
PROJECT_ID: ocid1.generativeaiproject.oc1.us-chicago-1.amaaaaaad6nji3aavcwaoww5f3igxhz47daipitc7qskhl6k7pw7lxtiy7jq
MODEL_ID: openai.gpt-5.2
MEMORY_SUBJECT_ID: copa-user-001


## 2) Criar conversa A para armazenar memoria

Criamos uma conversa com `memory_subject_id` e politica `store_only`,
depois conferimos o metadata salvo no servidor.


In [3]:
conv_a_metadata = {
    "memory_subject_id": MEMORY_SUBJECT_ID,
    "memory_access_policy": "store_only",
}

conv_a = client.conversations.create(metadata=conv_a_metadata)
conv_a_id = conv_a.id
conv_a_check = client.conversations.retrieve(conv_a_id)
conv_a_check_dict = conv_a_check.to_dict() if hasattr(conv_a_check, "to_dict") else json.loads(conv_a_check.model_dump_json())

print(json.dumps({
    "conversation_id": conv_a_id,
    "requested_metadata": conv_a_metadata,
    "stored_metadata": conv_a_check_dict.get("metadata", {}),
}, indent=2, ensure_ascii=False))


{
  "conversation_id": "conv_ord_qi9bmhocaaz2mzuw6o5jg4exu95me0q3pvk3iw9d0ls271bu",
  "requested_metadata": {
    "memory_subject_id": "copa-user-001",
    "memory_access_policy": "store_only"
  },
  "stored_metadata": {
    "memory_subject_id": "copa-user-001",
    "memory_access_policy": "store_only",
    "short_term_memory_optimization": "true"
  }
}


## 3) Semear fatos da Copa na conversa A

Guardamos 3 fatos de contexto. Isso alimenta short-term (mesma conversa)
e long-term (mesmo `memory_subject_id`).


In [4]:
copa_facts = [
    "Na fase de grupos da Copa, vitoria vale 3 pontos e empate vale 1.",
    "Em mata-mata, se empatar no tempo normal, pode haver prorrogacao e penaltis.",
    "Criterios comuns de desempate incluem saldo de gols e gols marcados.",
]

for fact in copa_facts:
    ack = client.responses.create(
        model=MODEL_ID,
        conversation=conv_a_id,
        input=f"Memorize este contexto para uso futuro do mesmo usuario: {fact}",
    )
    print("FACT:", fact)
    print("ACK:", getattr(ack, "output_text", "")[:180], "\n")


FACT: Na fase de grupos da Copa, vitoria vale 3 pontos e empate vale 1.
ACK: Não consigo “memorizar” informações de forma permanente entre conversas, mas posso usar esse contexto enquanto esta conversa estiver aberta.

Contexto registrado para esta conversa 

FACT: Em mata-mata, se empatar no tempo normal, pode haver prorrogacao e penaltis.
ACK: Não consigo memorizar isso permanentemente para futuras conversas, mas posso manter e usar enquanto esta conversa estiver aberta.

Contexto registrado para esta conversa: **em mata 

FACT: Criterios comuns de desempate incluem saldo de gols e gols marcados.
ACK: Não consigo memorizar permanentemente para futuras conversas, mas vou manter e usar enquanto esta conversa estiver aberta.

Contexto registrado para esta conversa: **critérios comu 



## 4) Validar short-term (mesma conversa A)

Perguntamos na mesma conversa. Se vier os fatos, short-term esta funcionando.


In [5]:
short_query = "Liste objetivamente as regras da Copa que eu acabei de te passar."

short_resp = client.responses.create(
    model=MODEL_ID,
    conversation=conv_a_id,
    input=short_query,
)

short_answer = getattr(short_resp, "output_text", "")

print(json.dumps({
    "conversation_id": conv_a_id,
    "query": short_query,
    "answer": short_answer,
}, indent=2, ensure_ascii=False))


{
  "conversation_id": "conv_ord_qi9bmhocaaz2mzuw6o5jg4exu95me0q3pvk3iw9d0ls271bu",
  "query": "Liste objetivamente as regras da Copa que eu acabei de te passar.",
  "answer": "- **Fase de grupos:** vitória vale **3 pontos**; empate vale **1 ponto**.  \n- **Mata-mata:** se houver **empate no tempo normal**, pode haver **prorrogação** e **pênaltis**.  \n- **Desempate (critérios comuns):** **saldo de gols** e **gols marcados**."
}


## 5) Validar long-term (conversa B nova, mesmo subject)

Criamos nova conversa com `recall_only`, aguardamos alguns segundos
para ingestao de memoria e validamos a recuperacao.


In [6]:
conv_b_metadata = {
    "memory_subject_id": MEMORY_SUBJECT_ID,
    "memory_access_policy": "recall_only",
}

time.sleep(8)

conv_b = client.conversations.create(metadata=conv_b_metadata)
conv_b_id = conv_b.id
conv_b_check = client.conversations.retrieve(conv_b_id)
conv_b_check_dict = conv_b_check.to_dict() if hasattr(conv_b_check, "to_dict") else json.loads(conv_b_check.model_dump_json())

long_query = "Quais informacoes voce lembra sobre regras da Copa que eu compartilhei antes?"
long_resp = client.responses.create(
    model=MODEL_ID,
    conversation=conv_b_id,
    input=long_query,
)
long_answer = getattr(long_resp, "output_text", "")

hit_count = 0
for fact in copa_facts:
    if fact.lower().replace(".", "") in long_answer.lower():
        hit_count += 1

print(json.dumps({
    "conversation_id": conv_b_id,
    "conversation_metadata": conv_b_check_dict.get("metadata", {}),
    "query": long_query,
    "answer": long_answer,
    "matched_facts": hit_count,
}, indent=2, ensure_ascii=False))


{
  "conversation_id": "conv_ord_u59092d64ehzkr30gxonjgm5ehmpv4qntzjsarql12sol5gd",
  "conversation_metadata": {
    "memory_subject_id": "copa-user-001",
    "memory_access_policy": "recall_only",
    "short_term_memory_optimization": "true"
  },
  "query": "Quais informacoes voce lembra sobre regras da Copa que eu compartilhei antes?",
  "answer": "Eu lembro que você compartilhou explicitamente esta regra para eu guardar:\n\n- **Fase de grupos da Copa**: **vitória vale 3 pontos** e **empate vale 1 ponto**.\n\nNos registros que tenho, não aparece você tendo me passado outras regras com o mesmo nível de confirmação. (Houve uma resposta anterior que mencionou mata-mata/prorrogação/pênaltis e critérios de desempate, mas isso não consta como algo que você tenha fornecido para memorizar.)",
  "matched_facts": 0
}


## 6) DDG como tool da LLM (OpenAI-compatible)

Agora a propria LLM decide quando chamar a tool `ddg_search`.
Implementamos a function local e fechamos o loop de tool calling com `responses.create`.


In [7]:
def ddg_search(query: str, max_results: int = 5):
    with DDGS() as ddgs:
        items = list(ddgs.text(query, max_results=max_results))

    results = []
    for item in items:
        results.append({
            "title": item.get("title", ""),
            "url": item.get("href", ""),
            "snippet": item.get("body", ""),
        })

    return {"query": query, "results": results}


ddg_tool = {
    "type": "function",
    "name": "ddg_search",
    "description": "Busca na web com DuckDuckGo e retorna titulo, url e resumo.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Termo de busca"},
            "max_results": {"type": "integer", "minimum": 1, "maximum": 10, "default": 5},
        },
        "required": ["query"],
    },
}

web_question = "Quais sao os criterios de desempate da fase de grupos da Copa 2026? Cite fontes."

resp = client.responses.create(
    model=MODEL_ID,
    conversation=conv_b_id,
    input=web_question,
    tools=[ddg_tool],
)

while True:
    calls = [
        item for item in resp.output
        if getattr(item, "type", "") == "function_call" and getattr(item, "name", "") == "ddg_search"
    ]

    if not calls:
        break

    tool_outputs = []
    for call in calls:
        args = json.loads(call.arguments or "{}")
        result = ddg_search(
            query=args.get("query", ""),
            max_results=int(args.get("max_results", 5)),
        )
        tool_outputs.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(result, ensure_ascii=False),
        })

    resp = client.responses.create(
        model=MODEL_ID,
        conversation=conv_b_id,
        input=tool_outputs,
        tools=[ddg_tool],
    )

print("Pergunta:", web_question)
print("\nResposta final:")
print(resp.output_text)




Pergunta: Quais sao os criterios de desempate da fase de grupos da Copa 2026? Cite fontes.

Resposta final:
A **FIFA ainda publica o “FIFA World Cup 26™ Regulations”** (regulamento oficial do torneio) mais perto do evento; até lá, o que existe de mais consistente/confirmável em fontes públicas é a **ordem de desempate que vem sendo divulgada para 2026** por veículos que citam o formato/tie-breakers do torneio.

## Critérios de desempate na fase de grupos (2026)
Quando duas ou mais seleções terminam **empatadas em pontos** no grupo, a ordem divulgada é:

1) **Mais pontos nos jogos entre as equipes empatadas** (confronto direto)  
2) **Melhor saldo de gols nos jogos entre as equipes empatadas**  
3) **Mais gols marcados nos jogos entre as equipes empatadas**  
4) **Melhor saldo de gols em todos os jogos do grupo**  
5) **Mais gols marcados em todos os jogos do grupo**  
6) **Melhor pontuação de fair play (disciplina)**  
7) **Sorteio / critério final definido pela FIFA** (quando persisti